# 03 — Transformer Architecture

Define a minimal decoder-only (GPT-style) transformer for next-token
prediction over the 3-state alphabet.  Walk through the architecture,
count parameters, and verify the forward pass.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Model

Architecture: token embedding → learned positional embedding →
N causal TransformerEncoder layers (pre-norm) → linear head over vocab.

This is a standard GPT-style setup.  The causal attention mask ensures
position *t* only attends to positions 0 … t, so the model predicts the
next token from its causal history — the same information used to define
the Markov chain.

In [ ]:
class AttentionOnlyBlock(nn.Module):
    """Causal self-attention block without a feedforward sublayer."""

    def __init__(self, d_model: int, nhead: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=0.0,
            batch_first=True,
        )

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        out, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        return x + out


class MarkovTransformer(nn.Module):
    """Minimal transformer: 2 attention layers + 1 shared MLP."""

    def __init__(
        self,
        vocab_size: int = 3,
        d_model: int = 32,
        nhead: int = 4,
        num_attention_layers: int = 2,
        dim_feedforward: int = 64,
        max_len: int = 64,
        use_token_embedding: bool = True,
        use_pos_embedding: bool = True,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.use_token_embedding = use_token_embedding
        self.use_pos_embedding = use_pos_embedding

        if use_token_embedding:
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            self.token_projection = None
        else:
            self.token_embedding = None
            self.token_projection = nn.Linear(vocab_size, d_model, bias=False)

        if use_pos_embedding:
            self.pos_embedding = nn.Embedding(max_len, d_model)
        else:
            self.pos_embedding = None

        self.attention_blocks = nn.ModuleList(
            [
                AttentionOnlyBlock(d_model=d_model, nhead=nhead)
                for _ in range(num_attention_layers)
            ]
        )

        self.mlp_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Linear(dim_feedforward, d_model),
        )
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, L = x.shape

        if self.use_token_embedding:
            h = self.token_embedding(x)
        else:
            x_one_hot = F.one_hot(x, num_classes=self.vocab_size).float()
            h = self.token_projection(x_one_hot)

        if self.use_pos_embedding:
            positions = torch.arange(L, device=x.device)
            h = h + self.pos_embedding(positions)

        causal_mask = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device), diagonal=1
        )
        for block in self.attention_blocks:
            h = block(h, attn_mask=causal_mask)

        h = h + self.mlp(self.mlp_norm(h))
        return self.head(h)

    def predict_probs(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward(x).softmax(dim=-1)

## Instantiate & inspect

In [ ]:
# Default config — kept small so the experiment runs on CPU
VOCAB_SIZE = 3
D_MODEL = 32
NHEAD = 4
NUM_ATTENTION_LAYERS = 2
DIM_FEEDFORWARD = 64
MAX_LEN = 64
USE_TOKEN_EMBEDDING = True
USE_POS_EMBEDDING = True

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_attention_layers=NUM_ATTENTION_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    max_len=MAX_LEN,
    use_token_embedding=USE_TOKEN_EMBEDDING,
    use_pos_embedding=USE_POS_EMBEDDING,
)
print(model)

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

# Per-module breakdown
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>6,} params")

## Smoke test

In [ ]:
# Verify output shapes for a random batch
batch_size, seq_len = 4, 16
x = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len))
logits = model(x)
print(f"Input  shape: {x.shape}")
print(f"Output shape: {logits.shape}  (expected: {(batch_size, seq_len, VOCAB_SIZE)})")
assert logits.shape == (batch_size, seq_len, VOCAB_SIZE)

# Softmax probabilities at position 0 should sum to 1 for each batch element
probs0 = logits[:, 0, :].softmax(dim=-1)
assert torch.allclose(probs0.sum(dim=-1), torch.ones(batch_size), atol=1e-5)
print("✓ Output shapes and probability sums are correct")